# Gaza building-damage classification — RSP ResNet-50 siamese pipeline

Fine-tunes a **ResNet-50 pretrained on MillionAID** (Remote Sensing Pretraining, [ViTAE-Transformer/RSP](https://github.com/ViTAE-Transformer/RSP)) as a shared-weight siamese encoder on pre/post-event PlanetScope RGB patches (32×32 px, 3 m) to classify building damage. The RSP checkpoint is a PyTorch state dict, so the model/training half of the pipeline is written in PyTorch; the data loading, spatial split, threshold selection and evaluation are unchanged from the EfficientNet version.

**Key evaluation choices**

1. **Spatial holdout** — the city is cut into three contiguous bands along its principal axis (train / val / test), with a buffer strip at each boundary wider than one patch footprint (96 m). Neighboring patches overlap on the ground, so a random split would leak near-duplicate pixels into the test set and inflate scores.
2. **Threshold tuning** — the decision threshold is swept on the *validation* set (F1 / balanced accuracy / Youden's J) and the best one is applied — once — to the test set.
3. **No test leakage anywhere** — normalization percentiles and class weights are computed from the train split only; the test set is touched exactly once, at the end.

**Backbone notes**

- The RSP file (`rsp-resnet-50-ckpt.pth`) is a full training checkpoint (`{'model', 'optimizer', 'lr_scheduler', 'epoch', 'config', ...}`); only the `'model'` entry is used and the 51-class MillionAID head is dropped. RSP's `resnet.py` uses the torchvision layer names, so the weights load 1:1 into `torchvision.models.resnet50`.
- RSP was pretrained on 224×224 RGB normalized with ImageNet mean/std, so the encoder upsamples the 32 px patch and applies that same normalization internally. `RESIZE_TO` controls the upsampling size (128 by default; 224 matches pretraining exactly at ~3× the compute).
- Two-phase fine-tuning as before: head only with a frozen backbone, then `layer3`+`layer4` unfrozen at a low LR. BatchNorm layers stay frozen throughout (batches of 32 are too small to re-estimate their statistics).
- Training mirrors the previous Keras setup: Adam, PR-AUC as the monitored metric, early stopping with best-weight restore, ReduceLROnPlateau, best-epoch checkpoint to Drive, fixed seeds.


In [ ]:
# ── Colab setup ──────────────────────────────────────────────────────────────
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running in Colab - adjust paths below.")

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
NPZ_PATH     = "/content/drive/MyDrive/War_Damage/datasets/Gaza_20240503_320a78e597.npz"
PARQUET_PATH = "/content/drive/MyDrive/War_Damage/datasets/Gaza_20240503_320a78e597.parquet"
OUT_DIR      = "/content/drive/MyDrive/War_Damage/models"
RSP_CKPT     = f"{OUT_DIR}/rsp-resnet-50-ckpt.pth"   # RSP-ResNet-50-E300 (MillionAID)

SEED         = 42
BATCH_SIZE   = 32
NUM_WORKERS  = 2
RESIZE_TO    = 128          # encoder input size (bilinear upsampling from 32px).
                            # RSP was pretrained at 224; 128 is a 4x upsample and
                            # leaves a 4x4 map before pooling. Use 224 to match
                            # pretraining exactly (about 3x the compute).

# Spatial split: contiguous bands along the city's principal axis
TRAIN_FRAC, VAL_FRAC = 0.60, 0.20        # test gets the remainder (0.20)
PATCH_SIZE_M = 32 * 3                     # 32 px @ 3 m/px PlanetScope = 96 m
BUFFER_M     = PATCH_SIZE_M + 10          # dead zone at split boundaries -> no
                                          # patch can straddle two splits

# Threshold selection: metric optimized on the *validation* set
# options: "f1", "balanced_accuracy", "youden" (tpr - fpr)
THRESHOLD_METRIC = "f1"

import os
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(RSP_CKPT), f"RSP checkpoint not found at {RSP_CKPT}"


In [ ]:
# ── Imports & reproducibility ────────────────────────────────────────────────
import json
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (precision_recall_curve, roc_curve, roc_auc_score,
                             average_precision_score, f1_score, precision_score,
                             recall_score, balanced_accuracy_score,
                             confusion_matrix, classification_report)

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"           # mixed precision on GPU
print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


## 1 · Load data and building coordinates
The parquet provides one row per building (same order as the `.npz`); its coordinates drive the spatial split.

In [ ]:
# ── Load imagery, labels and building coordinates ────────────────────────────
with np.load(NPZ_PATH) as data:
    images        = data["X"]                              # (N, 14, 32, 32)
    labels        = data["y"].astype(np.float32)           # (N,) binary 0/1
    channel_names = [n.strip() for n in data["channel_names"].tolist()]

buildings = pd.read_parquet(PARQUET_PATH)

# The parquet rows must align 1:1 with the npz rows - hard requirement for the
# spatial split. If this assert fires, the two files are not the same export.
assert len(buildings) == len(labels), (
    f"Row mismatch: parquet has {len(buildings)} rows, npz has {len(labels)}")

print(f"N = {len(labels)} | positive rate = {labels.mean():.3f}")
print(f"Channels: {channel_names}")
print(f"Parquet columns: {list(buildings.columns)}")


def extract_coords(df):
    """Return (x, y) arrays of building locations, trying common layouts."""
    cols = {c.lower(): c for c in df.columns}
    for cx, cy in [("longitude", "latitude"), ("lon", "lat"), ("lng", "lat"),
                   ("x", "y"), ("centroid_x", "centroid_y"),
                   ("center_x", "center_y")]:
        if cx in cols and cy in cols:
            return (df[cols[cx]].to_numpy(float),
                    df[cols[cy]].to_numpy(float), f"columns {cx}/{cy}")
    # geometry column (WKB bytes, WKT strings, or shapely objects)
    for gname in ("geometry", "geom", "wkb_geometry"):
        if gname in cols:
            from shapely import wkb, wkt
            def centroid(g):
                if isinstance(g, (bytes, bytearray)):
                    g = wkb.loads(bytes(g))
                elif isinstance(g, str):
                    g = wkt.loads(g)
                c = g.centroid
                return c.x, c.y
            xy = np.array([centroid(g) for g in df[cols[gname]]])
            return xy[:, 0], xy[:, 1], f"'{cols[gname]}' centroids"
    raise ValueError("No coordinate columns found. Available columns: "
                     f"{list(df.columns)} - add the right names to extract_coords().")


x_raw, y_raw, coord_src = extract_coords(buildings)
print(f"Coordinates from {coord_src}: "
      f"x∈[{x_raw.min():.4f}, {x_raw.max():.4f}], "
      f"y∈[{y_raw.min():.4f}, {y_raw.max():.4f}]")

# Convert lon/lat degrees to approximate local meters (equirectangular).
# If the values are already projected (magnitudes > 360), keep them as-is.
if np.abs(x_raw).max() <= 360 and np.abs(y_raw).max() <= 90:
    lat0 = np.deg2rad(y_raw.mean())
    x_m = (x_raw - x_raw.mean()) * 111_320 * np.cos(lat0)
    y_m = (y_raw - y_raw.mean()) * 110_540
else:
    x_m, y_m = x_raw - x_raw.mean(), y_raw - y_raw.mean()

## 2 · Spatial holdout split

In [ ]:
# ── Spatial holdout split ────────────────────────────────────────────────────
# Neighboring 32x32 patches overlap on the ground, so a random split leaks
# nearly-identical pixels between train and test. Instead we cut the city into
# three contiguous bands along its principal (longest) axis, and drop a buffer
# strip at each boundary so no patch can appear (even partially) in two splits.

XY = np.stack([x_m, y_m], axis=1)
XY_c = XY - XY.mean(axis=0)
_, _, Vt = np.linalg.svd(XY_c, full_matrices=False)
t = XY_c @ Vt[0]                       # position along the principal axis (m)

b1 = np.quantile(t, TRAIN_FRAC)                    # train | val boundary
b2 = np.quantile(t, TRAIN_FRAC + VAL_FRAC)         # val | test boundary
half = BUFFER_M / 2.0

split = np.full(len(t), "buffer", dtype=object)
split[t <  b1 - half] = "train"
split[(t >= b1 + half) & (t < b2 - half)] = "val"
split[t >= b2 + half] = "test"

idx_train = np.where(split == "train")[0]
idx_val   = np.where(split == "val")[0]
idx_test  = np.where(split == "test")[0]

print(f"{'split':<8}{'n':>6}{'pos rate':>10}")
for name, idx in [("train", idx_train), ("val", idx_val), ("test", idx_test),
                  ("buffer", np.where(split == "buffer")[0])]:
    pr = labels[idx].mean() if len(idx) else float("nan")
    print(f"{name:<8}{len(idx):>6}{pr:>10.3f}")

for name, idx in [("val", idx_val), ("test", idx_test)]:
    if len(np.unique(labels[idx])) < 2:
        print(f"WARNING: {name} split contains a single class - "
              "swap band order or adjust fractions.")

# Sanity check: minimum distance between any train and any test building
from scipy.spatial import cKDTree
d, _ = cKDTree(XY[idx_train]).query(XY[idx_test], k=1)
print(f"\nMin train↔test distance: {d.min():.0f} m "
      f"(patch footprint {PATCH_SIZE_M} m -> no overlap possible: {d.min() > PATCH_SIZE_M})")

In [ ]:
# ── Visualize the split ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 6), sharex=True, sharey=True)

colors = {"train": "tab:blue", "val": "tab:orange",
          "test": "tab:green", "buffer": "lightgray"}
for name, c in colors.items():
    m = split == name
    axes[0].scatter(x_m[m], y_m[m], s=4, c=c, label=f"{name} ({m.sum()})")
axes[0].legend(markerscale=3)
axes[0].set_title("Spatial split (contiguous bands + buffer)")

axes[1].scatter(x_m[labels == 0], y_m[labels == 0], s=4, c="silver", label="intact")
axes[1].scatter(x_m[labels == 1], y_m[labels == 1], s=4, c="crimson", label="damaged")
axes[1].legend(markerscale=3)
axes[1].set_title("Damage labels")

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")
plt.tight_layout()
plt.show()

## 3 · Preprocessing and PyTorch data loaders

In [ ]:
# ── Extract pre/post RGB and normalize (stats from TRAIN split only) ─────────
pre_ch  = [channel_names.index(c) for c in ["ps_pre_R",  "ps_pre_G",  "ps_pre_B"]]
post_ch = [channel_names.index(c) for c in ["ps_post_R", "ps_post_G", "ps_post_B"]]

# channels-last float32, R-G-B order
pre_imgs  = np.transpose(images[:, pre_ch,  :, :], (0, 2, 3, 1)).astype(np.float32)
post_imgs = np.transpose(images[:, post_ch, :, :], (0, 2, 3, 1)).astype(np.float32)
pre_imgs  = np.nan_to_num(pre_imgs,  nan=0.0, posinf=0.0, neginf=0.0)
post_imgs = np.nan_to_num(post_imgs, nan=0.0, posinf=0.0, neginf=0.0)

# Robust per-channel 2-98 percentile clip. Computing the percentiles on the
# train split only avoids leaking val/test statistics into preprocessing.
train_stack = np.concatenate([pre_imgs[idx_train], post_imgs[idx_train]], axis=0)
lo = np.percentile(train_stack, 2,  axis=(0, 1, 2)).astype(np.float32)
hi = np.percentile(train_stack, 98, axis=(0, 1, 2)).astype(np.float32)
del train_stack
print(f"Per-channel clip range:\n  lo = {lo}\n  hi = {hi}")

def normalize(x):
    x = np.clip(x, lo, hi)
    return (x - lo) / np.maximum(hi - lo, 1e-6)

pre_imgs, post_imgs = normalize(pre_imgs), normalize(post_imgs)

In [ ]:
# ── Dataset / DataLoader with pair-consistent augmentation ───────────────────
class PairDataset(Dataset):
    """Yields (pre, post, label) as CHW float tensors in [0, 1].
    Augmentation (train only): identical geometric transform for pre and post;
    mild independent brightness jitter to mimic acquisition differences."""

    def __init__(self, indices, training=False):
        # NHWC numpy -> NCHW torch, done once up front (the arrays fit in RAM)
        self.pre  = torch.from_numpy(pre_imgs[indices]).permute(0, 3, 1, 2).contiguous()
        self.post = torch.from_numpy(post_imgs[indices]).permute(0, 3, 1, 2).contiguous()
        self.y    = torch.from_numpy(labels[indices])
        self.training = training

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        pre, post = self.pre[i], self.post[i]
        if self.training:
            both = torch.cat([pre, post], dim=0)                     # (6, 32, 32)
            if torch.rand(()) < 0.5:
                both = both.flip(-1)                                 # left-right
            if torch.rand(()) < 0.5:
                both = both.flip(-2)                                 # up-down
            both = torch.rot90(both, int(torch.randint(0, 4, ())), dims=(-2, -1))
            pre, post = both[:3], both[3:]
            pre  = (pre  + (torch.rand(()) * 0.10 - 0.05)).clamp(0.0, 1.0)
            post = (post + (torch.rand(()) * 0.10 - 0.05)).clamp(0.0, 1.0)
        return pre, post, self.y[i]


def make_loader(indices, training=False):
    g = torch.Generator()
    g.manual_seed(SEED)
    return DataLoader(PairDataset(indices, training),
                      batch_size=BATCH_SIZE, shuffle=training,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"),
                      generator=g)


train_loader = make_loader(idx_train, training=True)
val_loader   = make_loader(idx_val)      # no shuffle -> prediction order == idx order
test_loader  = make_loader(idx_test)
val_y, test_y = labels[idx_val], labels[idx_test]

# Class imbalance from the train split. Keras' class_weight {0: w0, 1: w1}
# with w = N / (2 * n_class) is equivalent (up to a constant factor, which Adam
# ignores) to BCEWithLogitsLoss(pos_weight = w1 / w0 = n_neg / n_pos).
n_pos = float(labels[idx_train].sum())
n_neg = float(len(idx_train) - n_pos)
pos_weight = torch.tensor([n_neg / n_pos], device=DEVICE)
print(f"train: {int(n_neg)} intact / {int(n_pos)} damaged -> pos_weight = {pos_weight.item():.3f}")


## 4 · Model

In [ ]:
# ── Load the RSP ResNet-50 backbone (MillionAID pretraining) ────────────────
def _load_rsp_checkpoint(ckpt_path):
    """torch.load an RSP checkpoint without needing its training dependencies.

    RSP pickles the yacs config object into the checkpoint alongside the
    weights, so unpickling it normally requires `yacs` to be installed. We only
    want the tensors, so we register throwaway stand-ins for any such module for
    the duration of the load. Nothing from the config is used afterwards."""
    import sys, types

    class _Stub(dict):
        # Must raise AttributeError (not return None) for unknown attributes:
        # pickle probes instances for __setstate__/__reduce__, and a None return
        # is treated as a found method and then called -> TypeError.
        def __init__(self, *a, **k):
            super().__init__(*(a[:1] or ({},)))
        def __getattr__(self, k):
            try:
                return self[k]
            except KeyError:
                raise AttributeError(k)

    stubbed = []
    for name in ("yacs", "yacs.config"):
        if name not in sys.modules:
            m = types.ModuleType(name)
            m.CfgNode = _Stub
            m.__getattr__ = lambda _n, _S=_Stub: _S      # any other class -> _Stub
            sys.modules[name] = m
            stubbed.append(name)
    try:
        # weights_only=False: the file holds non-tensor objects (config, optimizer
        # state). Fine for a checkpoint you downloaded yourself and stored on Drive.
        return torch.load(ckpt_path, map_location="cpu", weights_only=False)
    finally:
        for name in stubbed:
            del sys.modules[name]


def load_rsp_resnet50(ckpt_path):
    """torchvision ResNet-50 initialised from the RSP MillionAID checkpoint.

    RSP's Scene Recognition/models/resnet.py uses torchvision's layer naming
    (conv1 / bn1 / layer1..4 / fc), so the tensors load 1:1. The .pth is a full
    training checkpoint ({'model', 'optimizer', 'lr_scheduler', 'epoch',
    'config', ...}); we keep only 'model' and drop the 51-class scene head."""
    ckpt = _load_rsp_checkpoint(ckpt_path)
    sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    sd = {(k[7:] if k.startswith("module.") else k): v for k, v in sd.items()}  # DDP prefix
    fc_shape = tuple(sd["fc.weight"].shape) if "fc.weight" in sd else None
    sd = {k: v for k, v in sd.items() if not k.startswith("fc.")}

    net = torchvision.models.resnet50(weights=None)
    net.fc = nn.Identity()                                # -> (B, 2048) pooled features
    missing, unexpected = net.load_state_dict(sd, strict=False)
    assert not unexpected, f"Unexpected keys in checkpoint: {unexpected[:5]}"
    assert not missing,    f"Missing keys (architecture mismatch?): {missing[:5]}"
    print(f"Loaded RSP ResNet-50: {len(sd)} tensors | epoch {ckpt.get('epoch', '?')} "
          f"| dropped head fc{fc_shape} | MillionAID acc@1 {ckpt.get('max_accuracy', '?')}")
    return net


# The original RSP file bundles the optimizer state and a yacs config object.
# Stripping it to just the tensors makes it ~3x smaller, loadable with the safe
# weights_only=True path, and free of any dependency on RSP's training code.
RSP_WEIGHTS = f"{OUT_DIR}/rsp-resnet-50-weights.pth"

if not os.path.exists(RSP_WEIGHTS):
    _ck = _load_rsp_checkpoint(RSP_CKPT)
    torch.save({"model": dict(_ck["model"]),
                "epoch": int(_ck.get("epoch", -1)),
                "max_accuracy": float(_ck.get("max_accuracy", float("nan")))},
               RSP_WEIGHTS)
    del _ck
    print(f"Wrote {RSP_WEIGHTS} ({os.path.getsize(RSP_WEIGHTS)/1e6:.1f} MB)")

# Verify it loads on the safe path, then point the pipeline at it
_chk = torch.load(RSP_WEIGHTS, map_location="cpu", weights_only=True)
print(f"weights_only=True load OK | {len(_chk['model'])} tensors | epoch {_chk['epoch']}")
del _chk
RSP_CKPT = RSP_WEIGHTS

In [ ]:
# ── Siamese model: shared RSP ResNet-50 (MillionAID-pretrained) encoder ──────
IMAGENET_MEAN = (0.485, 0.456, 0.406)    # RSP pretraining used timm's
IMAGENET_STD  = (0.229, 0.224, 0.225)    # IMAGENET_DEFAULT_MEAN / STD on [0,1] RGB
FEAT_DIM      = 2048


class SiameseDamage(nn.Module):
    """Shared encoder on pre & post, head on [f_pre, f_post, |f_pre - f_post|]."""

    def __init__(self, backbone, resize_to=RESIZE_TO, feat_dim=FEAT_DIM):
        super().__init__()
        self.backbone  = backbone
        self.resize_to = resize_to
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(3 * feat_dim, 256), nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, 1))

    def encode(self, x):
        x = F.interpolate(x, size=(self.resize_to, self.resize_to),
                          mode="bilinear", align_corners=False)
        x = (x - self.mean) / self.std
        return self.backbone(x)

    def forward(self, pre, post):
        # One backbone pass over the concatenated pair batch. Safe because BN is
        # always in eval mode here (frozen stats), so pre/post never mix.
        f = self.encode(torch.cat([pre, post], dim=0))
        f_pre, f_post = f.chunk(2, dim=0)
        x = torch.cat([f_pre, f_post, (f_pre - f_post).abs()], dim=1)
        return self.head(x).squeeze(1)                     # logits


def set_backbone_trainable(model, mode):
    """mode: 'frozen' | 'top' (layer3 + layer4) | 'all'.
    BatchNorm parameters are always frozen (their running stats are frozen
    separately by freeze_bn() at train time)."""
    for name, p in model.backbone.named_parameters():
        if mode == "frozen":
            p.requires_grad = False
        elif mode == "top":
            p.requires_grad = name.startswith(("layer3", "layer4"))
        else:
            p.requires_grad = True
    for m in model.backbone.modules():
        if isinstance(m, nn.BatchNorm2d):
            for p in m.parameters():
                p.requires_grad = False


def freeze_bn(model):
    """Keep backbone BatchNorm in eval mode (call right after model.train())."""
    for m in model.backbone.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()


def count_params(model):
    tot = sum(p.numel() for p in model.parameters())
    tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return tot, tr


model = SiameseDamage(load_rsp_resnet50(RSP_CKPT)).to(DEVICE)
set_backbone_trainable(model, "frozen")
tot, tr = count_params(model)
print(f"Parameters: {tot/1e6:.2f}M total | {tr/1e6:.2f}M trainable (phase 1)")

# Shape check on one batch
pre_b, post_b, y_b = next(iter(val_loader))
with torch.no_grad():
    model.eval()
    out = model(pre_b.to(DEVICE), post_b.to(DEVICE))
print(f"batch in: {tuple(pre_b.shape)} x2 -> logits {tuple(out.shape)}")

CKPT_PATH = f"{OUT_DIR}/siamese_rsp_resnet50_gaza_best.pt"


# ── Training utilities (mirror the previous Keras callbacks) ─────────────────
@torch.no_grad()
def predict(model, loader):
    """Sigmoid probabilities, in loader order (loaders are unshuffled)."""
    model.eval()
    probs = []
    for pre, post, _ in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(pre.to(DEVICE, non_blocking=True),
                           post.to(DEVICE, non_blocking=True))
        probs.append(torch.sigmoid(logits.float()).cpu())
    return torch.cat(probs).numpy()


def val_metrics(probs, y, thr=0.5):
    pred = (probs >= thr).astype(int)
    return {"auc_pr":    average_precision_score(y, probs),
            "auc_roc":   roc_auc_score(y, probs),
            "acc":       float((pred == y).mean()),
            "precision": precision_score(y, pred, zero_division=0),
            "recall":    recall_score(y, pred, zero_division=0)}


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    freeze_bn(model)
    total, n = 0.0, 0
    for pre, post, y in loader:
        pre  = pre.to(DEVICE, non_blocking=True)
        post = post.to(DEVICE, non_blocking=True)
        y    = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(pre, post)
        loss = criterion(logits.float(), y)               # loss in fp32
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total += loss.item() * len(y)
        n += len(y)
    return total / n


def fit(model, epochs, lr, tag, patience=8, lr_patience=4, lr_factor=0.5, min_lr=1e-6):
    """Adam + EarlyStopping(val_auc_pr, restore best) + ReduceLROnPlateau +
    best-epoch checkpoint to CKPT_PATH. Returns the epoch history as a DataFrame."""
    params    = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(params, lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=lr_factor, patience=lr_patience, min_lr=min_lr)
    scaler    = torch.amp.GradScaler(enabled=USE_AMP)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    history, best, best_state, bad = [], -1.0, None, 0
    for ep in range(1, epochs + 1):
        t0 = time.time()
        loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        m = val_metrics(predict(model, val_loader), val_y)
        scheduler.step(m["auc_pr"])
        cur_lr = optimizer.param_groups[0]["lr"]
        history.append({"phase": tag, "epoch": ep, "loss": loss, "lr": cur_lr,
                        **{f"val_{k}": v for k, v in m.items()}})
        flag = ""
        if m["auc_pr"] > best:
            best, bad, flag = m["auc_pr"], 0, "  *best*"
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, CKPT_PATH)
        else:
            bad += 1
        print(f"[{tag}] ep {ep:>2}/{epochs} | loss {loss:.4f} | "
              f"val auc_pr {m['auc_pr']:.4f} auc_roc {m['auc_roc']:.4f} "
              f"acc {m['acc']:.3f} P {m['precision']:.3f} R {m['recall']:.3f} | "
              f"lr {cur_lr:.1e} | {time.time() - t0:.0f}s{flag}")
        if bad >= patience:
            print(f"Early stopping (no val auc_pr improvement for {patience} epochs).")
            break

    model.load_state_dict(best_state)                     # restore_best_weights
    print(f"[{tag}] best val auc_pr = {best:.4f}")
    return pd.DataFrame(history)


## 5 · Training (two phases)

In [ ]:
# ── Phase 1: train the head with a frozen backbone ───────────────────────────
set_backbone_trainable(model, "frozen")
print(f"trainable params: {count_params(model)[1]/1e6:.2f}M")
hist1 = fit(model, epochs=20, lr=1e-3, tag="phase1")


In [ ]:
# ── Phase 2: unfreeze the top of the backbone, fine-tune at low LR ───────────
# layer3 + layer4 of ResNet-50 (~ the top half of the parameters); conv1/layer1/
# layer2 stay frozen and BatchNorm stays frozen everywhere.
set_backbone_trainable(model, "top")
print(f"trainable params: {count_params(model)[1]/1e6:.2f}M")
hist2 = fit(model, epochs=30, lr=1e-5, tag="phase2")

# Training curves across both phases
hist = pd.concat([hist1, hist2], ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist["loss"].values); axes[0].set_title("train loss")
axes[1].plot(hist["val_auc_pr"].values, label="val PR-AUC")
axes[1].plot(hist["val_auc_roc"].values, label="val ROC-AUC"); axes[1].legend()
for ax in axes:
    ax.axvline(len(hist1) - 0.5, ls=":", c="gray"); ax.set_xlabel("epoch")
plt.tight_layout(); plt.show()


## 6 · Threshold selection (validation set)

In [ ]:
# ── Threshold selection on the VALIDATION set ────────────────────────────────
# The 0.5 default is arbitrary, especially with class weighting. We sweep
# thresholds on the val set, pick the one maximizing THRESHOLD_METRIC, and
# only then touch the test set (once, with that fixed threshold).
val_probs = predict(model, val_loader)     # val_y defined with the loaders

grid = np.linspace(0.01, 0.99, 197)
def sweep(metric_fn):
    return np.array([metric_fn(val_y, (val_probs >= th).astype(int)) for th in grid])

scores = {
    "f1":                sweep(lambda y, p: f1_score(y, p, zero_division=0)),
    "balanced_accuracy": sweep(balanced_accuracy_score),
}
fpr, tpr, roc_th = roc_curve(val_y, val_probs)
# map Youden's J onto the same grid for a uniform interface
youden_on_grid = np.interp(grid, roc_th[::-1], (tpr - fpr)[::-1])
scores["youden"] = youden_on_grid

best_thr = float(grid[np.argmax(scores[THRESHOLD_METRIC])])
print(f"Best threshold by {THRESHOLD_METRIC} (on val): {best_thr:.3f}")
for name, s in scores.items():
    th = grid[np.argmax(s)]
    print(f"  argmax {name:<18}-> thr={th:.3f}, score={s.max():.3f}")

plt.figure(figsize=(7, 4.5))
for name, s in scores.items():
    plt.plot(grid, s, label=name)
plt.axvline(best_thr, ls="--", c="k", label=f"chosen ({best_thr:.3f})")
plt.axvline(0.5, ls=":", c="gray", label="default 0.5")
plt.xlabel("threshold"); plt.ylabel("val score"); plt.legend()
plt.title("Threshold sweep (validation set)")
plt.tight_layout(); plt.show()

## 7 · Test evaluation (spatially held out, evaluated once)

In [ ]:
# ── Final evaluation on the spatially held-out TEST set ──────────────────────
test_probs = predict(model, test_loader)

print(f"Test ROC-AUC: {roc_auc_score(test_y, test_probs):.4f}")
print(f"Test PR-AUC:  {average_precision_score(test_y, test_probs):.4f}")

for thr, tag in [(best_thr, f"chosen thr={best_thr:.3f}"), (0.5, "default thr=0.500")]:
    pred = (test_probs >= thr).astype(int)
    print(f"\n─── {tag} ───")
    print(f"F1: {f1_score(test_y, pred):.4f} | "
          f"balanced acc: {balanced_accuracy_score(test_y, pred):.4f}")
    print("Confusion matrix [[TN FP][FN TP]]:")
    print(confusion_matrix(test_y, pred))
    print(classification_report(test_y, pred,
                                target_names=["intact", "damaged"], digits=3))

# ROC and PR curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fpr, tpr, _ = roc_curve(test_y, test_probs)
axes[0].plot(fpr, tpr); axes[0].plot([0, 1], [0, 1], ls=":", c="gray")
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("Test ROC")
prec, rec, _ = precision_recall_curve(test_y, test_probs)
axes[1].plot(rec, prec)
axes[1].axhline(test_y.mean(), ls=":", c="gray", label="prevalence")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Test Precision-Recall"); axes[1].legend()
plt.tight_layout(); plt.show()

# Where do the errors sit on the map? Spatial error clusters usually mean the
# model is picking up neighborhood context rather than building damage.
pred = (test_probs >= best_thr).astype(int)
err = pred != test_y.astype(int)
plt.figure(figsize=(6, 6))
plt.scatter(x_m[idx_test][~err], y_m[idx_test][~err], s=6, c="silver", label="correct")
plt.scatter(x_m[idx_test][err],  y_m[idx_test][err],  s=10, c="crimson", label="error")
plt.gca().set_aspect("equal"); plt.legend(markerscale=2)
plt.title("Test-set errors in space"); plt.tight_layout(); plt.show()

## 8 · Save model + inference config

In [ ]:
# ── Save model + everything needed to reproduce inference ────────────────────
MODEL_PATH = f"{OUT_DIR}/siamese_rsp_resnet50_gaza_final.pt"
torch.save({"state_dict": model.state_dict(),
            "resize_to": RESIZE_TO, "feat_dim": FEAT_DIM,
            "arch": "SiameseDamage(torchvision resnet50, fc=Identity)"}, MODEL_PATH)

# The threshold and normalization stats are part of the model contract:
# without them, predictions on new data are not reproducible.
inference_config = {
    "framework": "pytorch",
    "model_path": MODEL_PATH,
    "backbone": "RSP ResNet-50 (MillionAID pretraining, ViTAE-Transformer/RSP)",
    "backbone_init_ckpt": RSP_CKPT,
    "decision_threshold": best_thr,
    "threshold_metric": THRESHOLD_METRIC,
    "norm_lo": lo.tolist(),
    "norm_hi": hi.tolist(),
    "imagenet_mean": list(IMAGENET_MEAN),
    "imagenet_std": list(IMAGENET_STD),
    "pre_channels":  ["ps_pre_R", "ps_pre_G", "ps_pre_B"],
    "post_channels": ["ps_post_R", "ps_post_G", "ps_post_B"],
    "resize_to": RESIZE_TO,
    "split": {"type": "spatial_bands_principal_axis",
              "train_frac": TRAIN_FRAC, "val_frac": VAL_FRAC,
              "buffer_m": BUFFER_M, "seed": SEED},
}
with open(f"{OUT_DIR}/inference_config.json", "w") as f:
    json.dump(inference_config, f, indent=2)

print(f"Saved model -> {MODEL_PATH}")
print(f"Saved inference config -> {OUT_DIR}/inference_config.json")
print(json.dumps(inference_config, indent=2))

# Reload check: the saved state dict already contains the fine-tuned backbone,
# so the original RSP checkpoint is NOT needed at inference time.
_bb = torchvision.models.resnet50(weights=None); _bb.fc = nn.Identity()
_m = SiameseDamage(_bb, resize_to=RESIZE_TO).to(DEVICE)
_m.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE)["state_dict"])
_p = predict(_m, test_loader)
print(f"Reload check: max |Δprob| vs in-memory model = {np.abs(_p - test_probs).max():.2e}")
